# Évaluation NLP : trois tâches, trois protocoles
**El Mehdi FATAHLLAH — Master Data & IA**

Résumé automatique, Question Answering et génération de texte.

Dans Colab : **Fichier > Importer un notebook**, puis **Exécution > Tout exécuter**. CPU suffisant. Aucune installation, aucun téléchargement, aucune clé API.

Nous construisons trois petits systèmes réellement exécutables. Ce sont des baselines pédagogiques sur corpus synthétique, pas des modèles préentraînés ni un benchmark. Les scores servent à comprendre les métriques.


## 0. Normalisation et métriques communes
Nous conservons les accents et les articles français. La ponctuation et la casse disparaissent. Les occurrences répétées comptent : l’intersection utilise des multiensembles (`Counter`). Les scores sont entre 0 et 1.

Précision = recouvrement / longueur de la prédiction. Rappel = recouvrement / longueur de la référence. F1 = moyenne harmonique. La normalisation diffère du script officiel anglais SQuAD, qui supprime aussi certains articles anglais.

In [1]:
import re, math, random, json
from collections import Counter, defaultdict

def tokens(text):
    return re.findall(r"[^\W_]+", text.lower(), flags=re.UNICODE)

def prf(overlap, pred_n, ref_n):
    p = overlap / pred_n if pred_n else 0.0
    r = overlap / ref_n if ref_n else 0.0
    return {"P": p, "R": r, "F1": 2*p*r/(p+r) if p+r else 0.0}

def ngrams(ts, n):
    return [tuple(ts[i:i+n]) for i in range(len(ts)-n+1)]

def rouge_n(pred, ref, n=1):
    a, b = Counter(ngrams(tokens(pred), n)), Counter(ngrams(tokens(ref), n))
    return prf(sum((a & b).values()), sum(a.values()), sum(b.values()))

def rouge_l(pred, ref):
    a, b = tokens(pred), tokens(ref)
    previous = [0]*(len(b)+1)
    for x in a:
        row = [0]
        for j,y in enumerate(b, 1):
            row.append(previous[j-1]+1 if x==y else max(previous[j],row[-1]))
        previous = row
    return prf(previous[-1],len(a),len(b))

print("Normalisation :", tokens("À Rabat !"))

Normalisation : ['à', 'rabat']


## 1. Résumé automatique : construire puis évaluer
Le résumé extractif sélectionne une phrase existante. Notre baseline retient la phrase qui contient le plus de mots du titre. Le titre est une entrée disponible au système, jamais la référence.

ROUGE signifie **Recall-Oriented Understudy for Gisting Evaluation**. ROUGE-1 compare les unigrammes, ROUGE-2 les bigrammes et ROUGE-L la plus longue sous-séquence commune. Ici, on rapporte P, R et F1 pour chaque variante, avec LCS au niveau du résumé entier, sans stemming.

In [2]:
def summarize(text, title):
    sentences = [s.strip() for s in re.split(r"(?<=[.!?])\s+", text) if s.strip()]
    if not sentences:
        return ""
    keywords = set(tokens(title))
    return max(sentences, key=lambda s: len(set(tokens(s)) & keywords))

summary_cases = [
    {"title":"Bloquer sa carte", "text":"La banque lance une application mobile. Elle permet de bloquer sa carte. Les agences restent ouvertes.", "ref":"L’application permet de bloquer sa carte."},
    {"title":"Formation données", "text":"Le master organise une conférence. Une formation sur les données commence lundi. Les inscriptions ferment vendredi.", "ref":"Une formation sur les données commence lundi."},
    {"title":"Train Rabat", "text":"Le train pour Rabat part à 8 h. Il arrive à 9 h. Le billet coûte 40 dirhams.", "ref":"Le train pour Rabat part à 8 h et arrive à 9 h."}
]
summary_results=[]
for item in summary_cases:
    pred=summarize(item['text'],item['title'])
    scores={"ROUGE-1":rouge_n(pred,item['ref'],1),"ROUGE-2":rouge_n(pred,item['ref'],2),"ROUGE-L":rouge_l(pred,item['ref'])}
    summary_results.append(scores)
    print("Résumé :",pred,"\nRéférence :",item['ref'])
    print(json.dumps(scores,ensure_ascii=False,indent=2))
print("Moyennes macro F1 :", {k:round(sum(r[k]['F1'] for r in summary_results)/len(summary_results),3) for k in summary_results[0]})

Résumé : Elle permet de bloquer sa carte. 
Référence : L’application permet de bloquer sa carte.
{
  "ROUGE-1": {
    "P": 0.8333333333333334,
    "R": 0.7142857142857143,
    "F1": 0.7692307692307692
  },
  "ROUGE-2": {
    "P": 0.8,
    "R": 0.6666666666666666,
    "F1": 0.7272727272727272
  },
  "ROUGE-L": {
    "P": 0.8333333333333334,
    "R": 0.7142857142857143,
    "F1": 0.7692307692307692
  }
}
Résumé : Une formation sur les données commence lundi. 
Référence : Une formation sur les données commence lundi.
{
  "ROUGE-1": {
    "P": 1.0,
    "R": 1.0,
    "F1": 1.0
  },
  "ROUGE-2": {
    "P": 1.0,
    "R": 1.0,
    "F1": 1.0
  },
  "ROUGE-L": {
    "P": 1.0,
    "R": 1.0,
    "F1": 1.0
  }
}
Résumé : Le train pour Rabat part à 8 h. 
Référence : Le train pour Rabat part à 8 h et arrive à 9 h.
{
  "ROUGE-1": {
    "P": 1.0,
    "R": 0.6153846153846154,
    "F1": 0.761904761904762
  },
  "ROUGE-2": {
    "P": 1.0,
    "R": 0.5833333333333334,
    "F1": 0.7368421052631579
  },
  "R

### Limite : les mots peuvent cacher une contradiction
Un résumé erroné peut conserver presque tous les mots. Un synonyme correct peut au contraire perdre des points. La vérification humaine de la fidélité factuelle reste nécessaire. BERTScore serait un complément sémantique fondé sur des embeddings, sans constituer une preuve de vérité. Il n'est pas calculé dans ce notebook sans dépendances.

In [3]:
reference = "Le train part à 8 h."
for candidate in ["Le train part à 9 h.", "Départ du train à huit heures."]:
    print(candidate, "ROUGE-1 :", round(rouge_n(candidate,reference)['F1'],3))

Le train part à 9 h. ROUGE-1 : 0.833
Départ du train à huit heures. ROUGE-1 : 0.333


## 2. Question Answering : extraction et abstention
Le contexte contient une ville et une heure. La baseline utilise des règles explicites pour extraire les réponses. Une question inconnue produit une chaîne vide, notre convention d’abstention.

**Exact Match (EM)** vaut 1 si les tokens normalisés coïncident, sinon 0. **Token F1** mesure le recouvrement. Pour plusieurs références valides, on prend le maximum par métrique, puis la moyenne sur les questions. Ce protocole s'inspire de SQuAD, avec une normalisation française documentée et un exemple d'abstention.

In [4]:
def answer(context, question):
    q=" ".join(tokens(question))
    if "où" in q and "siège" in q:
        match=re.search(r"siège est à ([^.]+)", context, re.I)
    elif "heure" in q and "ouvre" in q:
        match=re.search(r"ouvre à ([^.]+)", context, re.I)
    else:
        return ""
    return match.group(1).strip() if match else ""

def exact_match(pred, ref):
    return float(tokens(pred)==tokens(ref))

def token_f1(pred, ref):
    a,b=tokens(pred),tokens(ref)
    if not a or not b:
        return float(a==b)
    return prf(sum((Counter(a)&Counter(b)).values()),len(a),len(b))['F1']

context="Le siège est à Rabat. L’agence ouvre à 9 h."
qa_cases=[("Où est le siège ?",["Rabat","à Rabat"]),
          ("À quelle heure l’agence ouvre ?",["9 h","9 heures"]),
          ("Qui dirige la banque ?",[""])]
qa_results=[]
for question,refs in qa_cases:
    pred=answer(context,question)
    em=max(exact_match(pred,r) for r in refs)
    f1=max(token_f1(pred,r) for r in refs)
    qa_results.append((em,f1))
    print(question, "=>", pred or "[ABSTENTION]", "EM=",em,"F1=",round(f1,3))
print("Moyennes :", {"EM":sum(x[0] for x in qa_results)/len(qa_results),"F1":sum(x[1] for x in qa_results)/len(qa_results)})
print("Exemple diapositive : à Rabat / Rabat", exact_match("à Rabat","Rabat"), token_f1("à Rabat","Rabat"))

Où est le siège ? => Rabat EM= 1.0 F1= 1.0
À quelle heure l’agence ouvre ? => 9 h EM= 1.0 F1= 1.0
Qui dirige la banque ? => [ABSTENTION] EM= 1.0 F1= 1.0
Moyennes : {'EM': 1.0, 'F1': 1.0}
Exemple diapositive : à Rabat / Rabat 0.0 0.6666666666666666


### Interprétation critique
Le score parfait de cette baseline sur trois cas simples reflète leur construction. Il ne démontre aucune compréhension générale. Ajouter des paraphrases, des lieux distracteurs et des questions impossibles révèle les faiblesses. Une abstention correcte ne doit pas masquer une tendance à s'abstenir sur des questions répondables.

In [5]:
challenge="Le siège est à Rabat. L’agence est à Casablanca."
print("Question ambiguë pour la règle :",answer(challenge,"Dans quelle ville se trouve le siège ?") or "[ABSTENTION]")

Question ambiguë pour la règle : [ABSTENTION]


## 3. Génération de texte : modèle bigramme de Markov
Le modèle apprend P(mot suivant | mot précédent) à partir d'un petit corpus d'entraînement. Un lissage additif (alpha=0,1) donne une probabilité non nulle à chaque token du vocabulaire. Les mots inconnus du test deviennent `<unk>`. Le vocabulaire vient uniquement de l'entraînement.

La génération est échantillonnée avec une graine fixe. La **perplexité** vaut exp(-moyenne des log-probabilités) sur un corpus de test distinct de l'entraînement, en incluant la fin de phrase. Nous comparons deux lissages à tokenizer, vocabulaire et test identiques.

In [6]:
train=["la banque propose une application mobile", "la banque protège les données", "le client utilise une application mobile", "le client bloque sa carte", "la banque bloque une carte", "une application protège les données", "le client consulte son compte", "la banque propose un compte"]
test=["le client utilise son compte", "la banque protège une carte"]
vocab=sorted({w for s in train for w in tokens(s)} | {"</s>","<unk>"})
counts=defaultdict(Counter)
for s in train:
    seq=["<s>"]+tokens(s)+["</s>"]
    for prev,nxt in zip(seq,seq[1:]): counts[prev][nxt]+=1

def probability(prev,nxt,alpha=0.1):
    return (counts[prev][nxt]+alpha)/(sum(counts[prev].values())+alpha*len(vocab))

def generate(seed=7,max_words=22,alpha=0.1):
    rng=random.Random(seed)
    prev="<s>"; result=[]
    for _ in range(max_words):
        nxt=rng.choices(vocab,weights=[probability(prev,w,alpha) for w in vocab],k=1)[0]
        if nxt=="</s>": break
        result.append(nxt); prev=nxt
    return " ".join(result)

def perplexity(corpus,alpha):
    logsum=0.; n=0
    for s in corpus:
        seq=["<s>"]+[w if w in vocab else "<unk>" for w in tokens(s)]+["</s>"]
        for a,b in zip(seq,seq[1:]):
            logsum+=math.log(probability(a,b,alpha)); n+=1
    return math.exp(-logsum/n) if n else float('nan')

def distinct(text,n):
    grams=ngrams(tokens(text),n)
    return len(set(grams))/len(grams) if grams else 0.

generated=[generate(seed) for seed in range(4)]
for text in generated:
    print("Texte :",text or "[VIDE]", "Distinct-2 :",round(distinct(text,2),3))
for alpha in [0.1,1.0]:
    print("alpha=",alpha,"perplexité TEST=",round(perplexity(test,alpha),3))
for text in ["banque banque banque banque", "carte nuage piano orange"]:
    print(text, "Distinct-2 :",distinct(text,2))

Texte : protège les données Distinct-2 : 1.0
Texte : la le consulte compte Distinct-2 : 1.0
Texte : une son <unk> <unk> son les données Distinct-2 : 1.0
Texte : la banque les données carte Distinct-2 : 1.0
alpha= 0.1 perplexité TEST= 4.284
alpha= 1.0 perplexité TEST= 9.789
banque banque banque banque Distinct-2 : 0.3333333333333333
carte nuage piano orange Distinct-2 : 1.0


### Ce que les scores ne prouvent pas
Distinct-2 = nombre de bigrammes uniques / nombre total de bigrammes, ici au niveau de chaque texte. Il dépend de la longueur : comparer des générations de longueur similaire. Une suite absurde peut obtenir 1. Une faible perplexité indique une meilleure prédiction du test avec ce protocole, pas une meilleure factualité.

Pour une génération libre, il existe beaucoup de réponses valides : les métriques à référence unique (BLEU ou ROUGE) sont insuffisantes. Une grille humaine doit distinguer cohérence, pertinence, diversité et fidélité aux informations disponibles. Ne pas inventer de notes humaines automatiques.

## 4. Vérifications des calculs
Les assertions contrôlent des cas dont la réponse est connue : identité, recouvrement partiel, répétition et normalisation des probabilités.

In [7]:
assert rouge_n("a b","a b",1)['F1']==1
assert rouge_n("a a","a",1)['P']==0.5
assert rouge_l("a b c","a c")['R']==1
assert exact_match("Rabat !","rabat")==1
assert abs(token_f1("à Rabat","Rabat")-2/3)<1e-10
assert token_f1("","")==1
assert distinct("banque banque banque",2)==0.5
assert abs(sum(probability("<s>",w) for w in vocab)-1)<1e-10
assert perplexity(test,0.1)>0
print("Toutes les vérifications passent.")

Toutes les vérifications passent.


## 5. Résumé et prolongements
| Tâche | Calcul réalisé | Complément indispensable |
|---|---|---|
| Résumé | ROUGE-1, ROUGE-2, ROUGE-L | Fidélité et couverture |
| QA | Exact Match, token F1 | Cas impossibles, robustesse |
| Génération | Perplexité sur test, Distinct-2 | Cohérence, pertinence, factualité |

Pour aller plus loin : remplacer les trois baselines par des modèles préentraînés tout en conservant ce protocole, élargir les jeux de test, faire annoter des exemples à plusieurs évaluateurs et mesurer l'accord.

### Sources
- Lin (2004), ROUGE : https://aclanthology.org/W04-1013/
- Rajpurkar et al. (2016), SQuAD : https://aclanthology.org/D16-1264/
- Li et al. (2016), diversité : https://aclanthology.org/N16-1014/
- Hugging Face, perplexité : https://huggingface.co/docs/transformers/en/perplexity

Les données françaises sont des exemples synthétiques créés pour ce cours. Aucun score n'est un résultat sur un benchmark public.
